# Batch run

Runs several configurations one after another in one session and analyses them at the end.
The default list is a quick 7B comparison of all formats (20 trials per bit position) plus a
persistent-fault check for the most critical bit of each format. About 45 minutes on an L4.

Use an **L4** runtime and *Runtime > Run all*. If the session dies, run all again; finished
trials are skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO = "https://github.com/mehmetd7mir/LLM-ScaleBit-SoftError.git"
CODE = "/content/LLM-ScaleBit-SoftError"
COMMIT = None   # set a commit hash to rerun with an exact code version

import os, subprocess, sys
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
if COMMIT:
    subprocess.run(["git", "-C", CODE, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)
sys.path.insert(0, CODE)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# Hugging Face token: needed for gated models (Llama). Add it once under the key icon
# (Colab Secrets) with the name HF_TOKEN.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("no HF_TOKEN secret (fine for Qwen models)")

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4"
gpu = torch.cuda.get_device_name(0)
print(gpu, f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
if "L4" not in gpu:
    print("WARNING: the study runs on L4. Do not mix GPU types inside one study.")
# ECC state of this GPU (the fault model does not depend on it; we report it in the paper)
try:
    ecc = subprocess.run(["nvidia-smi", "--query-gpu=ecc.mode.current", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    print("ECC:", ecc or "unknown")
except FileNotFoundError:
    print("ECC: nvidia-smi not found")

## Runs

Each entry is one configuration. `persist` is an optional `(class, bit, n_faults)`: one flip is
kept in place while every question is answered. FP16 exponent bit 4 is bit 14 of the word.

In [ ]:
RESULTS_ROOT = "/content/drive/MyDrive/LLM-ScaleBit-Results"
STUDY = "quick_7b"
MODEL = "Qwen/Qwen2.5-7B-Instruct"
GPTQ = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"
TASK = "mmlu"
N = 20                                   # trials per bit position
OVERRIDE = {"element": 20, "tensor_scale": 10}
FAULTS = 10                              # persistent faults per run

RUNS = [
    dict(fmt="fp",        dtype="float16",  persist=("exponent", 4, FAULTS)),
    dict(fmt="int4_rtn",  dtype="float16",  persist=("scale", 14, FAULTS)),
    dict(fmt="int4_gptq", dtype="float16",  persist=("scale", 14, FAULTS)),
    dict(fmt="mxfp4",     dtype="float16",  persist=("scale", 7, FAULTS)),
    dict(fmt="nvfp4",     dtype="float16",  persist=("tensor_scale", 30, FAULTS)),
    # dtype sub-study teaser: does the MXFP4 scale-MSB failure stay visible in BF16?
    dict(fmt="mxfp4",     dtype="bfloat16", persist=("scale", 7, FAULTS), classes=["scale"]),
]

## Run everything

In [ ]:
import gc, time, traceback
import torch
from scalebit.run import Config, Experiment

done, failed = [], []
for r in RUNS:
    t0 = time.time()
    tag = f"{r['fmt']}/{r['dtype']}"
    print(f"\n========== {tag} ==========")
    ex = None
    try:
        cfg = Config(model_id=MODEL, fmt=r["fmt"], task=TASK, dtype=r["dtype"],
                     gptq_id=GPTQ if r["fmt"] == "int4_gptq" else None,
                     trials_per_bit=N, trials_override=dict(OVERRIDE),
                     classes=r.get("classes"), study=STUDY, results_root=RESULTS_ROOT)
        ex = Experiment(cfg).setup()
        ex.golden()
        if not (ex.check_determinism() and ex.check_sanity()):
            raise RuntimeError("a check failed; skipping this run")
        ex.run()
        if r.get("persist"):
            ex.run_persistent(*r["persist"])
        done.append(tag)
    except Exception:
        traceback.print_exc()
        failed.append(tag)
    finally:
        del ex
        gc.collect()
        torch.cuda.empty_cache()
    print(f"{tag}: {time.time() - t0:.0f} s")
print("\ndone:", done)
print("failed:", failed)

## Results

In [ ]:
from scalebit import analysis
from IPython.display import display
import pandas as pd
pd.set_option("display.width", 200)

t = analysis.report(f"{RESULTS_ROOT}/{STUDY}")
display(t["environment"])
cols = ["format", "dtype", "class", "n", "fail", "sdc", "visible", "avf", "avf_lo", "avf_hi"]
display(t["avf_by_class"][cols])

In [ ]:
pos = t["avf_by_position"]
display(pos[pos.fail > 0][["format", "dtype", "class", "bit", "role", "n", "fail", "sdc", "visible", "avf"]])

In [ ]:
cols = ["format", "dtype", "bits_per_weight", "fit", "fit_lo", "fit_hi", "metadata_bit_share",
        "metadata_fit_share", "metadata_fit_share_min", "fit_vs_fp", "fit_vs_fp_worst"]
print("FIT, as measured")
display(t["fit"][cols])
print("FIT if bits that are constant in each tensor were forced back to their value")
display(t["fit_constant_bits_protected"][["format", "dtype", "fit", "fit_hi"]])

In [ ]:
print("Persistent faults: one flip kept in place, every question answered")
display(t["persistent"])